In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
df_model = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "sales_features.parquet")
df_model.shape

(898000, 20)

## 1. Time-Series Cross-Validation
Kenapa tidak boleh pakai train_test_split biasa atau K-Fold acak: kalau split random, model bisa dilatih pakai data bulan Desember untuk memprediksi bulan Juni — itu “mengintip masa depan”. Validasi harus selalu: latih di masa lalu, uji di masa depan, meniru kondisi nyata saat model dipakai.

In [2]:
from sklearn.model_selection import TimeSeriesSplit

feature_cols = [c for c in df_model.columns if c not in ["date", "sales"]]
cat_features = ["store", "item", "day_of_week", "month", "week_of_year"]

for col in cat_features:
    df_model[col] = df_model[col].astype("category")

X = df_model[feature_cols]
y = df_model["sales"]
dates = df_model["date"]

# Urutkan berdasarkan tanggal dulu -- wajib untuk time-series split
sort_idx = dates.argsort()
X = X.iloc[sort_idx].reset_index(drop=True)
y = y.iloc[sort_idx].reset_index(drop=True)
dates_sorted = dates.iloc[sort_idx].reset_index(drop=True)

tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    train_start, train_end = dates_sorted.iloc[train_idx[0]], dates_sorted.iloc[train_idx[-1]]
    val_start, val_end = dates_sorted.iloc[val_idx[0]], dates_sorted.iloc[val_idx[-1]]
    print(f"Fold {fold}: train {train_start.date()} s/d {train_end.date()} "
          f"({len(train_idx)} baris) | val {val_start.date()} s/d {val_end.date()} ({len(val_idx)} baris)")

Fold 0: train 2013-01-31 s/d 2013-11-26 (149670 baris) | val 2013-11-26 s/d 2014-09-21 (149666 baris)
Fold 1: train 2013-01-31 s/d 2014-09-21 (299336 baris) | val 2014-09-21 s/d 2015-07-18 (149666 baris)
Fold 2: train 2013-01-31 s/d 2015-07-18 (449002 baris) | val 2015-07-18 s/d 2016-05-12 (149666 baris)
Fold 3: train 2013-01-31 s/d 2016-05-12 (598668 baris) | val 2016-05-12 s/d 2017-03-07 (149666 baris)
Fold 4: train 2013-01-31 s/d 2017-03-07 (748334 baris) | val 2017-03-07 s/d 2017-12-31 (149666 baris)


## 2. Model ML dengan LightGBM

In [3]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

def wmape(y_true, y_pred):
    """Weighted MAPE -- lebih stabil dari MAPE biasa saat ada nilai sales mendekati 0."""
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100

cv_results = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    train_set = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_features)
    val_set = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_features, reference=train_set)

    params = {
        "objective": "regression",
        "metric": "mae",
        "num_leaves": 63,
        "learning_rate": 0.05,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
        "verbosity": -1,
        "seed": 42,
    }

    model = lgb.train(
        params, train_set, num_boost_round=500,
        valid_sets=[val_set], callbacks=[lgb.early_stopping(30, verbose=False)]
    )

    pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, pred)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    wmape_val = wmape(y_val.values, pred)

    cv_results.append({"fold": fold, "mae": mae, "rmse": rmse, "wmape": wmape_val})
    print(f"Fold {fold}: MAE={mae:.3f}, RMSE={rmse:.3f}, WMAPE={wmape_val:.2f}%")

cv_df = pd.DataFrame(cv_results)
print(f"\nRata-rata CV -- MAE: {cv_df['mae'].mean():.3f}, RMSE: {cv_df['rmse'].mean():.3f}, WMAPE: {cv_df['wmape'].mean():.2f}%")

Fold 0: MAE=5.923, RMSE=7.784, WMAPE=12.03%
Fold 1: MAE=5.629, RMSE=7.334, WMAPE=11.28%
Fold 2: MAE=5.745, RMSE=7.485, WMAPE=11.15%
Fold 3: MAE=6.076, RMSE=7.933, WMAPE=10.69%
Fold 4: MAE=6.283, RMSE=8.152, WMAPE=10.09%

Rata-rata CV -- MAE: 5.931, RMSE: 7.737, WMAPE: 11.05%


WMAPE (Weighted MAPE) dipakai sebagai metrik tambahan karena hasilnya dalam bentuk persentase, jauh lebih mudah dikomunikasikan ke stakeholder non-teknis dibanding MAE mentah (“model meleset rata-rata 8%” lebih bisa dipahami daripada “MAE 12.4 unit”).